# Subir OpenAlex a tu bucket de GCS sin descargarlo

Es el mismo comando del curso (`gcloud storage cp`), pero el origen es un servidor público en vez de tu Drive. Los archivos pasan en memoria de OpenAlex a tu bucket; no se guardan en tu laptop, en Drive ni en el disco de Colab.

**Orden:** 0) configuración, 1) funciones, 2) ver entidades, 3) plan de `works`, 4) copiar `works`, 5) catálogos, 6) validar.

**No uses "Ejecutar todo".** Corre celda por celda y, antes de copiar, revisa que `PROYECTO` y `DESTINO` sean los tuyos.

## 0. Autenticación y configuración

In [ ]:
from google.colab import auth
auth.authenticate_user()

project_id = 'big-data-lunes-2026-1'     # <-- CAMBIA por TU proyecto
bucket_id  = 'big-data-lunes-20260223'   # <-- CAMBIA por TU bucket

!gcloud config set project {project_id}
# Revisa location y clase de almacenamiento: lo ideal es STANDARD y la misma región que tu Dataproc (us-central1)
!gcloud storage buckets describe gs://{bucket_id} | grep -iE "location|storage_class"


## 1. Funciones de ayuda (correr una vez)
Sondeo de tamaños con `HEAD`, plan de copia hasta *N* GB, copia en paralelo con reintentos y verificación de tamaños.

In [ ]:
import subprocess, shlex, time, re, requests
from concurrent.futures import ThreadPoolExecutor, as_completed

def human(n):
    for u in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024 or u == "TB":
            return f"{n:.1f} {u}"
        n /= 1024

def probe_sizes(urls, workers=16):
    """HEAD a cada URL para conocer su tamaño SIN descargar nada."""
    def head(u):
        for i in range(3):
            try:
                r = requests.head(u, allow_redirects=True, timeout=30)
                cl = r.headers.get("Content-Length")
                if r.ok and cl:
                    return u, int(cl)
            except requests.RequestException:
                pass
            time.sleep(1 + i)
        return u, None
    with ThreadPoolExecutor(workers) as ex:
        return dict(ex.map(head, urls))

def build_plan(items, target_gb):
    """items: lista de (url, nombre_destino, bytes). Toma archivos en orden hasta ~target_gb."""
    plan, total = [], 0
    for url, name, size in items:
        if size is None:
            continue
        if total >= target_gb * 1024**3:
            break
        plan.append((url, name, size))
        total += size
    print(f"Plan: {len(plan)} archivos, {human(total)} (objetivo {target_gb} GB)")
    return plan

def gcs_existing(dest_prefix):
    """{ruta_gs: bytes} de lo que ya existe bajo dest_prefix (permite reanudar sin repetir)."""
    r = subprocess.run(["gcloud", "storage", "ls", "-l", dest_prefix.rstrip("/") + "/**"],
                       capture_output=True, text=True)
    out = {}
    for line in r.stdout.splitlines():
        m = re.match(r"\s*(\d+)\s+\S+\s+(gs://\S+)$", line)
        if m:
            out[m.group(2)] = int(m.group(1))
    return out

def stream_one(url, dest, transform=None, retries=3):
    """URL -> (tubería en memoria) -> GCS. No escribe nada en disco ni en Drive."""
    q_url, q_dest = shlex.quote(url), shlex.quote(dest)
    curl = f"curl -sSfL --retry 5 --retry-delay 5 {q_url}"
    if transform == "zst2gz":      # re-comprime en vuelo: .zst -> .gz
        pipe = f"{curl} | zstd -dc | gzip -1 | gcloud storage cp - {q_dest}"
    else:
        pipe = f"{curl} | gcloud storage cp - {q_dest}"
    err = ""
    for i in range(retries):
        r = subprocess.run(["bash", "-o", "pipefail", "-c", pipe], capture_output=True, text=True)
        if r.returncode == 0:
            return True, ""
        err = r.stderr[-400:]
        time.sleep(5 * (i + 1))
    return False, err

def transfer(plan, dest_prefix, workers=8, transform=None):
    """Copia en paralelo. Es idempotente: si se corta, vuelve a correr y salta lo ya subido."""
    dest_prefix = dest_prefix.rstrip("/")
    rename = (lambda n: re.sub(r"\.zst$", ".gz", n)) if transform == "zst2gz" else (lambda n: n)
    existing = gcs_existing(dest_prefix)
    todo = []
    for url, name, size in plan:
        dest = f"{dest_prefix}/{rename(name)}"
        already = (dest in existing) if transform else (existing.get(dest) == size)
        if not already:
            todo.append((url, dest, size))
    print(f"Ya estaban: {len(plan) - len(todo)} | Por copiar: {len(todo)} ({human(sum(t[2] for t in todo))})")
    failed, moved, t0 = [], 0, time.time()
    with ThreadPoolExecutor(workers) as ex:
        futs = {ex.submit(stream_one, u, d, transform): (u, d, s) for u, d, s in todo}
        for i, f in enumerate(as_completed(futs), 1):
            u, d, s = futs[f]
            ok, err = f.result()
            if ok:
                moved += s
            else:
                failed.append((u, err))
            if i % 5 == 0 or i == len(todo):
                mins = (time.time() - t0) / 60
                print(f"[{i}/{len(todo)}] {human(moved)} origen procesado en {mins:.1f} min | fallas: {len(failed)}")
    if not transform:
        after = gcs_existing(dest_prefix)
        bad = [d for _, d, s in todo if after.get(d) != s]
        print("Verificación de tamaños:", "OK" if not bad else f"{len(bad)} archivos con tamaño distinto -> vuelve a correr la celda")
    if failed:
        print("FALLARON (vuelve a correr la celda para reintentar):")
        for u, e in failed[:5]:
            print(" -", u, "|", e.strip()[:150])
    return failed

def s3_list_folders(bucket, prefix=""):
    """Lista 'carpetas' de un bucket S3 PÚBLICO por HTTPS, sin credenciales de AWS."""
    r = requests.get(f"https://{bucket}.s3.amazonaws.com/",
                     params={"list-type": "2", "prefix": prefix, "delimiter": "/"}, timeout=30)
    r.raise_for_status()
    return re.findall(r"<CommonPrefixes><Prefix>([^<]+)</Prefix>", r.text)

def s3_list_objects(bucket, prefix):
    """Lista TODOS los objetos (clave, bytes) de un bucket S3 público, con paginación y sin credenciales."""
    out, token = [], None
    while True:
        params = {"list-type": "2", "prefix": prefix}
        if token:
            params["continuation-token"] = token
        r = requests.get(f"https://{bucket}.s3.amazonaws.com/", params=params, timeout=60)
        r.raise_for_status()
        for blk in re.findall(r"<Contents>(.*?)</Contents>", r.text, flags=re.S):
            k = re.search(r"<Key>([^<]+)</Key>", blk).group(1)
            s = int(re.search(r"<Size>(\d+)</Size>", blk).group(1))
            out.append((k, s))
        t = re.search(r"<NextContinuationToken>([^<]+)</NextContinuationToken>", r.text)
        if not t:
            break
        token = t.group(1)
    return out


## Receta 1 · OpenAlex (bucket público de AWS S3, sin cuenta de AWS)

Publicaciones científicas en JSON Lines `.gz`. Cada entidad (`works`, `authors`, `institutions`...) trae un `manifest.json` con la lista de archivos.
La licencia del bucket es **CC0** (dominio público: uso libre, incluso comercial) y no pide cuenta. Los objetos se alcanzan por HTTPS en `openalex.s3.amazonaws.com`. Release de septiembre 2026: ~626 millones de registros por formato; JSONL comprimido ≈ 745 GB (`works` ≈ 615 GB) y Parquet ≈ 770 GB.

- `FORMAT = "jsonl"` es el dato crudo (ideal para mostrar tu ETL JSON → Parquet). Si el clúster sufre, `"parquet"` (snappy) también está publicado oficialmente.
- Los archivos vienen particionados por `updated_date`, **no** por año de publicación ni por país. Con `MUESTREO_ALEATORIO = True` se toma una muestra repartida entre particiones; el filtro por tema o país lo harás después en Spark. Documenta este sesgo en tu sección *Limitaciones de los datos*.
- Para las entidades pequeñas (`institutions`, `topics`, `sources`, etc.) repite la celda cambiando `ENTITY` y subiendo `TARGET_GB`; traerá todo lo que haya.

In [ ]:
# Qué hay disponible (carpetas del bucket público)
print(s3_list_folders("openalex", "data/"))
print(s3_list_folders("openalex", "data/jsonl/"))
print(s3_list_folders("openalex", "data/parquet/"))


In [ ]:
import random

ENTITY    = "works"
FORMAT    = "jsonl"          # "jsonl" (crudo) o "parquet"
TARGET_GB = 25               # 25 GB de works (~36 archivos de ~700 MB). Los de la prueba de 1 GB se saltan solos
MUESTREO_ALEATORIO = True
BASE_HTTPS = "https://openalex.s3.amazonaws.com/"

items = []
# Intento 1: manifest (según el README del bucket: data/<formato>/manifest.json; puede haber uno por entidad)
for u in [f"{BASE_HTTPS}data/{FORMAT}/{ENTITY}/manifest.json", f"{BASE_HTTPS}data/{FORMAT}/manifest.json"]:
    try:
        r = requests.get(u, timeout=120)
        if not r.ok:
            continue
        m = r.json()
        for e in (m["entries"] if isinstance(m, dict) else m):
            if f"/{FORMAT}/{ENTITY}/" in e["url"]:
                size = (e.get("meta") or {}).get("content_length")
                items.append((e["url"].replace("s3://openalex/", BASE_HTTPS), e["url"].split(f"/{ENTITY}/", 1)[1], size))
        if items:
            print("Manifest usado:", u); break
    except Exception as ex:
        print("Manifest no utilizable:", u, "->", ex)

# Intento 2: listar el bucket directamente (también trae los tamaños)
if not items:
    print("Listando el bucket directamente...")
    for key, size in s3_list_objects("openalex", f"data/{FORMAT}/{ENTITY}/"):
        if key.endswith((".gz", ".parquet")):
            items.append((BASE_HTTPS + key, key.split(f"/{ENTITY}/", 1)[1], size))
assert items, "No encontré archivos. Revisa FORMAT/ENTITY con la celda que lista carpetas."
print("Archivos encontrados:", len(items))

if MUESTREO_ALEATORIO:
    random.seed(7); random.shuffle(items)

limit, plan, total = TARGET_GB * 1024**3, [], 0
for k in range(0, len(items), 64):                        # si falta el tamaño, se consulta con HEAD por lotes
    if total >= limit: break
    batch = items[k:k+64]
    sz = probe_sizes([u for u, _, s in batch if s is None]) or {}
    for u, n, s in batch:
        s = s if s is not None else sz.get(u)
        if s and total < limit:
            plan.append((u, n, int(s))); total += int(s)
print(f"Plan: {len(plan)} archivos, {human(total)}")


In [ ]:
DEST = f"gs://{bucket_id}/openalex/{FORMAT}/{ENTITY}"
print("PROYECTO:", project_id)
print("DESTINO :", DEST)   # <-- confirma que sea TU bucket antes de copiar
failed = transfer(plan, DEST, workers=8)


## Catálogos de OpenAlex (tablas pequeñas)
Instituciones, financiadores, temas, países, fuentes, editoriales y premios (~4.6 GB en total). No traemos `authors` (76.7 GB): `works` ya trae a los autores incrustados.

In [ ]:
BASE_HTTPS = "https://openalex.s3.amazonaws.com/"
print("PROYECTO:", project_id)
for e in ["institutions", "funders", "topics", "countries", "sources", "publishers", "awards"]:
    objs = [(k, s) for k, s in s3_list_objects("openalex", f"data/jsonl/{e}/") if k.endswith(".gz")]
    plan = [(BASE_HTTPS + k, k.split(f"/{e}/", 1)[1], s) for k, s in objs]
    print(f"\n== {e}: {len(plan)} archivos, {human(sum(p[2] for p in plan))}")
    print("DESTINO :", f"gs://{bucket_id}/openalex/jsonl/{e}")
    transfer(plan, f"gs://{bucket_id}/openalex/jsonl/{e}", workers=8)


## Validar lo que quedó en el bucket

In [ ]:
PREFIX = "openalex"      # o "ais", "gdelt"...
!gcloud storage du --summarize --readable-sizes gs://{bucket_id}/{PREFIX}
!gcloud storage ls -l gs://{bucket_id}/{PREFIX}/** | head -8


## Siguiente paso (flujo del curso)

`GCS (raw)` → clúster Dataproc con Hive/Spark → limpieza y escritura a **Parquet particionado** → tablas externas en BigQuery → Looker Studio.

Antes de procesar ≥ 20 GB comprimidos, redimensiona el clúster del script del profe (2 workers `n2-standard-2` es poco; el JSON o CSV se expande varias veces al descomprimir) y **borra el clúster al terminar** para cuidar tus créditos.